# Protocol 5 — Causal Neuromodulation: Anterior Insula and Frontoparietal Gating

Simulates Pred 5.A–5.E from Protocol 5, updated to **protocol specification v1.1.0**. Right-aINS stimulation
uses tFUS (Tier 1) or deep TMS (Tier 2) against a **modality-matched sham**. The cortical arms (dlPFC, PPC)
use figure-of-eight TMS against the registered TMS reference (vertex). PCI comes from **separate resting
pre/post single-pulse probes**, not from the task.

* **Pred 5.A** — PCI falls at aINS (~20 %) versus matched sham and at dlPFC/PPC (15–25 %) versus the TMS reference.
  HEP–PCI coupling is disrupted at aINS and preserved at dlPFC. Each effect must survive its within-modality comparison.
* **Pred 5.B** *(primary, α = 0.05)* — aINS stimulation drops HEP–P3b coupling from ≈ 0.40–0.50 to < 0.15,
  and exteroceptive P3b changes by < 10 %.
* **Pred 5.C / 5.D** — Continuous IA × site interaction on ΔPCI, present at aINS only.
  *Conditional on the unresolved IA-moderation multiplicity lock*; reported descriptively here.
* **Pred 5.E** — dlPFC null-on-HEP equivalence: 95 % HDI of the standardised paired active − matched-sham HEP
  effect lies wholly inside d ∈ [−0.15, +0.15], with BF₀₁ ≥ 6.

The secondary ΔPCI family (dlPFC vs reference, PPC vs reference, aINS vs matched sham) uses α = 0.0167 for each test.

> All data are synthetic. Stimulation parameters still need approved dosimetry; nothing here authorises a
> stimulation schedule. Tier 3 (pharmacological) is exploratory, needs separate registration, and is not simulated.
>
> **EP-5 | Paper 2 | Empirical | Depends on: EP-0 | Spec v1.1.0, pre-collection |
> N enrolled = 52, analysable = 44, minimum = 36; engagement fMRI subset = 10**

### What changed from v1.0

| Item | v1.0 notebook | v1.1 specification |
|---|---|---|
| N | 36 / 28 | 52 enrolled, 44 analysable (d = 0.5, α = 0.05, power 0.90), 36 minimum |
| Control | Vertex sham for all | aINS vs **modality-matched sham**; dlPFC/PPC vs TMS reference |
| PCI | Derived from task ignition rate | Separate resting pre/post probes, ≥ 150 clean pulses, PCI_max cutoff 0.31 |
| Trials | 480 | 4 blocks × 200 = 800 (160 visual + 40 heartbeat per session; 120 TMS / 40 no-TMS visual) |
| Pred 5.E | — | New: Bayesian ROPE equivalence for dlPFC HEP (±0.15, 95 % HDI, BF₀₁ ≥ 6) |
| Gate | — | Engagement gate (fMRI subset ≥ 10 % ROI reduction; < 20 % cohort failure) |
| κ, α, β | 100, 0.3, 0.7 | κ = `null`; no legacy α/β defaults |

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import integrate
from scipy.stats import pearsonr, ttest_rel
from scipy.stats import t as t_dist

from apgi.core import BETA_SM_DEFAULT, compute_pi_i_eff

In [ ]:
# Load the v1.1 protocol specification so every threshold below is read from
# the registered JSON rather than retyped by hand.
def load_protocol(filename):
    for root in (Path.cwd(), *Path.cwd().parents):
        path = root / "protocols" / filename
        if path.exists():
            return json.loads(path.read_text(encoding="utf-8"))
    raise FileNotFoundError(f"protocols/{filename} not found above {Path.cwd()}")


PROTO = load_protocol("protocol_5_causal_tms.json")
assert PROTO["version"] == "1.1.0", PROTO["version"]
PRED = {p["id"]: p for p in PROTO["sub_predictions"]}
print(f"{PROTO['protocol_id']}  spec v{PROTO['version']}  status={PROTO['study_status']}")
for pid, p in PRED.items():
    print(f"  {pid:<9} [{p['status']}]  {p['claim'][:78]}")


def verdict(confirmed, falsified=False):
    """v1.1 three-way decision: failing to confirm is not falsification."""
    if confirmed:
        return "CONFIRMED (synthetic)"
    if falsified:
        return "FALSIFIED (synthetic)"
    return "INDETERMINATE (not confirmed; falsification rule not met)"

In [ ]:
def jzs_bf10(t, n, r=0.707):
    """JZS Bayes factor for a one-sample / paired t-test (Rouder et al., 2009).

    Cauchy prior scale r = 0.707, as registered for the v1.1 Bayesian tests.
    The likelihood ratio is formed inside the integral to avoid underflow.
    """
    nu = n - 1

    def integrand(g):
        ratio = ((1 + t**2 / ((1 + n * g * r**2) * nu)) / (1 + t**2 / nu)) ** (-(nu + 1) / 2)
        prior = (2 * np.pi) ** -0.5 * g**-1.5 * np.exp(-1 / (2 * g))
        return (1 + n * g * r**2) ** -0.5 * ratio * prior

    return float(integrate.quad(integrand, 0, np.inf, limit=200)[0])

## 1 — Protocol 5 design (v1.1)

In [ ]:
P, BLOCKS, TRIALS = PROTO["participants"], PROTO["blocks"], PROTO["trial_structure"]
N_ENROLLED, N_ANALYSABLE = P["n_target"], P["n_analyzable_target"]  # 52, 44
N_VISUAL = BLOCKS["visual_trials_per_session"]       # 160
N_HEARTBEAT = BLOCKS["heartbeat_trials_per_session"] # 40
PCI_CUTOFF = PROTO["pci"]["cutoff"]                  # 0.31 (PCI_max)
GATE = PROTO["engagement_gate"]
THR_B = PRED["Pred 5.B"]["confirming_thresholds"]
THR_E = PRED["Pred 5.E"]["confirming_thresholds"]
ALPHA_SECONDARY = PROTO["power_analysis"]["secondary_alpha"]  # 0.0167

CONDITIONS = ["ains_sham", "ains_active", "vertex_reference", "dlpfc", "ppc"]
LABELS = {"ains_sham": "aINS sham", "ains_active": "aINS active", "vertex_reference": "Vertex (ref)",
          "dlpfc": "dlPFC", "ppc": "PPC"}

# Illustrative generative effects (synthetic; not registered)
DELTA_PI = {"ains_sham": 0.0, "ains_active": -0.40, "vertex_reference": 0.0, "dlpfc": 0.0, "ppc": 0.0}
COUPLING = {"ains_sham": 0.45, "ains_active": 0.08, "vertex_reference": 0.45, "dlpfc": 0.43, "ppc": 0.44}
PCI_DROP = {"ains_sham": 0.00, "ains_active": 0.20, "vertex_reference": 0.02, "dlpfc": 0.20, "ppc": 0.18}

print(f"N enrolled {N_ENROLLED}, analysable {N_ANALYSABLE}; per session {N_VISUAL} visual + {N_HEARTBEAT} heartbeat trials")
print(f"TMS SOA {TRIALS['tms_soa_ms']} ms at {TRIALS['tms_rmt_percent']}% RMT; PCI: {TRIALS['pci_measurement']}")
rng = np.random.default_rng(1)

## 2 — Engagement gate

In [ ]:
roi_reduction = rng.normal(16, 5, GATE["fmri_subset_n"])                 # % aINS ROI response reduction
eeg_exponent = rng.normal(22, 8, N_ANALYSABLE - GATE["fmri_subset_n"])   # % aperiodic-exponent change
fail = np.concatenate([roi_reduction < GATE["fmri_roi_response_reduction_percent_minimum"],
                       eeg_exponent < GATE["outside_fmri_subset_eeg_exponent_change_percent_minimum"]])
gate_ok = fail.mean() < GATE["cohort_failure_fraction_strict_maximum"]
print(f"fMRI subset: mean ROI reduction {roi_reduction.mean():.1f}% (criterion ≥ {GATE['fmri_roi_response_reduction_percent_minimum']}%)")
print(f"Cohort engagement failures: {fail.mean():.0%} (must be < {GATE['cohort_failure_fraction_strict_maximum']:.0%}) → "
      f"{'gate passed' if gate_ok else 'gate FAILED — Pred 5.A/5.B not evaluable'}")
print("EEG limitation:", GATE["eeg_limitation"])

## 3 — Simulate sessions

In [ ]:
def simulate_session(cond, ia_z, pci_base, rng):
    pi_i = max(0.05, np.exp(0.15 * ia_z) + DELTA_PI[cond])
    M_hat = rng.uniform(0.0, 0.5, N_HEARTBEAT)
    pi_eff = np.array([compute_pi_i_eff(pi_i, BETA_SM_DEFAULT, float(m)) for m in M_hat])
    hep = pi_eff + rng.normal(0, 0.10, N_HEARTBEAT)
    # Trial-level HEP–P3b coupling at the registered target strength for this site
    z = (hep - hep.mean()) / hep.std()
    rho = COUPLING[cond]
    p3b_intero = 5.0 + rho * z + np.sqrt(1 - rho**2) * rng.normal(0, 1, N_HEARTBEAT)
    p3b_extero = 5.0 + rng.normal(0, 1.0, N_VISUAL).mean() + rng.normal(0, 0.1)
    ia_drop = 0.04 * ia_z if cond == "ains_active" else 0.0  # high-Πⁱ participants lose more PCI at aINS
    pci_post = pci_base * (1 - PCI_DROP[cond] - ia_drop) + rng.normal(0, 0.015)
    return dict(hep=hep.mean(), coupling=pearsonr(hep, p3b_intero)[0], p3b_extero=p3b_extero,
                pci_pre=pci_base + rng.normal(0, 0.01), pci_post=pci_post)


IA_z = rng.normal(0, 1, N_ANALYSABLE)
pci_base = rng.normal(0.48, 0.04, N_ANALYSABLE)
subj_hep = rng.normal(0, 0.15, N_ANALYSABLE)
S = {c: [] for c in CONDITIONS}
for i in range(N_ANALYSABLE):
    for c in CONDITIONS:
        s = simulate_session(c, IA_z[i], pci_base[i], rng)
        s["hep"] += subj_hep[i]
        S[c].append(s)
S = {c: {k: np.array([s[k] for s in S[c]]) for k in S[c][0]} for c in CONDITIONS}
delta_pci = {c: (S[c]["pci_post"] - S[c]["pci_pre"]) / S[c]["pci_pre"] for c in CONDITIONS}
for c in CONDITIONS:
    print(f"{LABELS[c]:<13} HEP–P3b coupling r = {np.tanh(np.arctanh(S[c]['coupling']).mean()):.2f}   "
          f"ΔPCI = {delta_pci[c].mean():+.1%}   post-PCI = {S[c]['pci_post'].mean():.3f}")

## 4 — Pred 5.B (primary): aINS abolishes HEP–P3b coupling; exteroceptive P3b spared

In [ ]:
z_active, z_sham = np.arctanh(S["ains_active"]["coupling"]), np.arctanh(S["ains_sham"]["coupling"])
t_b, p_b = ttest_rel(z_active, z_sham)
r_active = np.tanh(z_active.mean())
extero_change = abs(S["ains_active"]["p3b_extero"].mean() / S["ains_sham"]["p3b_extero"].mean() - 1) * 100
confirmed_5B = (p_b < THR_B["primary_alpha"] and r_active < THR_B["hep_p3b_coupling_strict_maximum"]
                and extero_change < THR_B["exteroceptive_p3b_change_percent_strict_maximum"])
print(f"aINS active vs matched sham coupling: r = {r_active:.2f} vs {np.tanh(z_sham.mean()):.2f}, "
      f"t({N_ANALYSABLE - 1}) = {t_b:.2f}, p = {p_b:.1e}")
print(f"Exteroceptive P3b change = {extero_change:.1f}% (must be < 10%)")
print(f"Pred 5.B → {verdict(confirmed_5B)}")

## 5 — Pred 5.A: Site-specific PCI reductions (secondary ΔPCI family, α = 0.0167 each)

In [ ]:
family = [("dlpfc", "vertex_reference"), ("ppc", "vertex_reference"), ("ains_active", "ains_sham")]
family_ok = []
for active, ref in family:
    t, p = ttest_rel(delta_pci[active], delta_pci[ref])
    drop = -(delta_pci[active].mean() - delta_pci[ref].mean())
    family_ok.append(p < ALPHA_SECONDARY and t < 0)
    print(f"  {LABELS[active]:<12} vs {LABELS[ref]:<12} PCI reduction = {drop:.1%}  p = {p:.1e}")

# HEP–PCI coupling (participant level): disrupted at aINS, preserved at dlPFC
r_hp = {c: pearsonr(S[c]["hep"], S[c]["pci_post"])[0] for c in ["ains_active", "dlpfc"]}
print(f"  HEP–PCI r: aINS active = {r_hp['ains_active']:.2f}, dlPFC = {r_hp['dlpfc']:.2f}")
confirmed_5A = gate_ok and all(family_ok)
print(f"Pred 5.A → {verdict(confirmed_5A)}  (HEP–PCI dissociation contrast awaits the locked manifest)")

## 6 — Pred 5.E: dlPFC HEP equivalence (Bayesian ROPE)

The posterior for the paired mean difference under a reference prior is a scaled t distribution, so its
95 % HDI equals the t interval. We standardise it by the SD of the differences.

In [ ]:
diff = S["dlpfc"]["hep"] - S["vertex_reference"]["hep"]  # TMS reference stands in for the matched TMS sham
n = len(diff)
d_z = diff.mean() / diff.std(ddof=1)
half = t_dist.ppf(0.5 + THR_E["hdi_level"] / 2, n - 1) / np.sqrt(n)
hdi = (d_z - half, d_z + half)
t_e = diff.mean() / (diff.std(ddof=1) / np.sqrt(n))
bf01 = 1 / jzs_bf10(t_e, n)
lo, hi = THR_E["rope_d"]
confirmed_5E = lo < hdi[0] and hdi[1] < hi and bf01 >= THR_E["bf01_minimum"]
falsified_5E = hdi[1] < lo  # wholly outside ROPE in the HEP-suppressing direction
print(f"dlPFC HEP d_z = {d_z:+.3f}, 95% HDI [{hdi[0]:+.3f}, {hdi[1]:+.3f}], ROPE [{lo}, {hi}], BF₀₁ = {bf01:.2f}")
print(f"Pred 5.E → {verdict(confirmed_5E, falsified_5E)}")
n_needed = int(np.ceil((1.96 / hi) ** 2))
print(f"Design check: HDI half-width ≈ 1.96/√N, so even a true d = 0 needs N ≳ {n_needed} to fit inside ±{hi}; "
      f"max BF₀₁ at N = {n} is {1 / jzs_bf10(0.0, n):.2f}")

> **Design flag.** At the analysable N = 44, the 95 % HDI on d is about ±0.30 wide, twice the ROPE, and BF₀₁
> cannot exceed ≈ 6.1. Pred 5.E is therefore almost always indeterminate at the registered N. This matches
> the JSON note that equivalence power is *not established* by the superiority calculation.

## 7 — Pred 5.C / 5.D: IA × site moderation (descriptive; awaiting multiplicity lock)

In [ ]:
for c in ["ains_active", "dlpfc", "ppc"]:
    r, p = pearsonr(IA_z, delta_pci[c])
    print(f"  IA_z vs ΔPCI at {LABELS[c]:<12} r = {r:+.2f} (p = {p:.1e})")
print("Status:", PRED["Pred 5.C"]["evaluation_status"])

## 8 — Visualise

In [ ]:
colors = ["#AAAAAA", "#d6604d", "#CCCCCC", "#2166ac", "#9966FF"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

ax = axes[0]
ax.bar([LABELS[c] for c in CONDITIONS], [S[c]["pci_post"].mean() for c in CONDITIONS],
       yerr=[S[c]["pci_post"].std() / np.sqrt(N_ANALYSABLE) for c in CONDITIONS],
       color=colors, alpha=0.85, edgecolor="white", width=0.55, capsize=4)
ax.axhline(PCI_CUTOFF, ls="--", lw=1, color="k", alpha=0.6, label=f"PCI_max cutoff {PCI_CUTOFF}")
ax.set_ylabel("Post-stimulation resting PCI")
ax.set_title("Pred 5.A — PCI by site", fontsize=10)
ax.tick_params(axis="x", rotation=25, labelsize=8)
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
r_vals = [np.tanh(np.arctanh(S[c]["coupling"]).mean()) for c in CONDITIONS]
ax.bar([LABELS[c] for c in CONDITIONS], r_vals, color=colors, alpha=0.85, edgecolor="white", width=0.55)
ax.axhline(THR_B["hep_p3b_coupling_strict_maximum"], ls="--", lw=1, color="k", label="0.15 criterion")
ax.set_ylabel("HEP–P3b coupling (r)")
ax.set_title("Pred 5.B — interoceptive gating", fontsize=10)
ax.tick_params(axis="x", rotation=25, labelsize=8)
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[2]
ax.axvspan(lo, hi, color="#00CC99", alpha=0.25, label="ROPE ±0.15")
ax.errorbar([d_z], [0], xerr=[[d_z - hdi[0]], [hdi[1] - d_z]], fmt="o", color="#2166ac", capsize=6,
            label="95% HDI")
ax.axvline(0, color="k", lw=0.6, alpha=0.4)
ax.set_xlim(-0.6, 0.6)
ax.set_yticks([])
ax.set_xlabel("Standardised dlPFC HEP effect (d)")
ax.set_title(f"Pred 5.E — equivalence, BF₀₁ = {bf01:.1f}", fontsize=10)
ax.legend(fontsize=8)
ax.spines[["top", "right", "left"]].set_visible(False)

fig.suptitle(f"Protocol 5 — Causal neuromodulation · spec v1.1.0 · synthetic N = {N_ANALYSABLE}", fontsize=12)
plt.tight_layout()
plt.show()